# Train piper model on ilspeech

In [27]:
# Prepare dependencies

import os

!sudo apt-get install espeak-ng -y
!git clone https://github.com/thewh1teagle/piper -b hebrew
%cd piper/src/python

# Don't use uv venv but use global uv
os.environ["UV_CONSTRAINT"] = ""
os.environ["UV_BUILD_CONSTRAINT"] = ""
os.environ["UV_PRERELEASE"] = "if-necessary-or-explicit"
os.environ["UV_SYSTEM_PYTHON"] = "false"
!uv venv
!uv pip install -e .
!./build_monotonic_align.sh

/content
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
espeak-ng is already the newest version (1.50+dfsg-10ubuntu0.1).
0 upgraded, 0 newly installed, 0 to remove and 34 not upgraded.
Cloning into 'piper'...
remote: Enumerating objects: 2129, done.
remote: Counting objects: 100% (143/143), done.
remote: Compressing objects: 100% (26/26), done.
remote: Total 2129 (delta 124), reused 117 (delta 117), pack-reused 1986 (from 3)
Receiving objects: 100% (2129/2129), 214.04 MiB | 30.74 MiB/s, done.
Resolving deltas: 100% (1175/1175), done.
/content/piper/src/python
Using CPython 3.10.12 interpreter at: /usr/bin/python3.10
Creating virtual environment at: .venv
Activate with: source .venv/bin/activate
Resolved 66 packages in 1.37s
   Building piper-train @ file:///content/piper/src/python
   Building piper-train @ file:///content/piper/src/python
⠙ Preparing packages... (0/54)
   Building piper-train @ file:///content/piper/src/python
⠙ Prepar

In [ ]:
# Prepare dataset

!wget https://huggingface.co/datasets/thewh1teagle/ILSpeech/resolve/main/ilspeech_2025_04_v1.zip
!unzip ilspeech_2025_04_v1.zip

In [29]:
# Prepare checkpoint
!wget https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/en/en_US/ryan/medium/epoch=4641-step=3104302.ckpt

--2025-04-21 12:18:43--  https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/en/en_US/ryan/medium/epoch=4641-step=3104302.ckpt
Resolving huggingface.co (huggingface.co)... 3.166.152.110, 3.166.152.44, 3.166.152.65, ...
Connecting to huggingface.co (huggingface.co)|3.166.152.110|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://cdn-lfs.hf.co/repos/6c/3b/6c3bb73012be460f91a23801a60eb3831f76d24f6812c18e0ca05b16673ae99f/1dcd988ec37e97e47d765dead11e891bc4920e56efc6a7352f6492603fe41bf5?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27epoch%25253D4641-step%25253D3104302.ckpt%3B+filename%3D%22epoch%253D4641-step%253D3104302.ckpt%22%3B&Expires=1745241523&Policy=eyJTdGF0ZW1lbnQiOlt7IkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiQVdTOkVwb2NoVGltZSI6MTc0NTI0MTUyM319LCJSZXNvdXJjZSI6Imh0dHBzOi8vY2RuLWxmcy5oZi5jby9yZXBvcy82Yy8zYi82YzNiYjczMDEyYmU0NjBmOTFhMjM4MDFhNjBlYjM4MzFmNzZkMjRmNjgxMmMxOGUwY2EwNWIxNjY3M2FlOTlmLzFkY2Q5ODhlYzM3ZTk3ZTQ3ZD

In [34]:
import os

# Preprocess
!uv run python -m piper_train.preprocess \
    --language he \
    --input-dir ilspeech_2025_04_21_v1 \
    --output-dir ./train \
    --dataset-format ljspeech \
    --single-speaker \
    --sample-rate 22050 \
    --raw-phonemes

 build				 mypy.ini	        run-docker
 build_monotonic_align.sh	 piper_train	        scripts
 Dockerfile			 piper_train.egg-info   setup.py
'epoch=4641-step=3104302.ckpt'	 README.md	        train
 ilspeech_2025_04_21_v1		 requirements_dev.txt
 ilspeech_2025_04_v1.zip	 requirements.txt
INFO:preprocess:Single speaker dataset
INFO:preprocess:Wrote dataset config
INFO:preprocess:Processing 579 utterance(s) with 2 worker(s)


In [ ]:
# Train
!uv pip install torchmetrics==0.11.4
!uv run python -m piper_train \
        --dataset-dir "./train" \
        --accelerator 'gpu' \
        --devices 1 \
        --batch-size 32 \
        --validation-split 0 \
        --num-test-examples 0 \
        --max_epochs 90000 \
        --resume_from_checkpoint ./epoch=4641-step=3104302.ckpt \
        --checkpoint-epochs 1 \
        --precision 32

In [ ]:
# infer

!cat ../../etc/test_sentences/test_he.jsonl  | \
        python3 -m piper_train.infer \
            --sample-rate 22050 \
            --checkpoint ./train/lightning_logs/version_0/checkpoints/*.ckpt \
            --output-dir ./output